In [1]:
# --------------------------------- Part 1: Imports ---------------------------------
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import time
import pickle
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, auc, accuracy_score)
from scipy.stats import ttest_rel
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, Conv1D, Flatten, concatenate, Dropout,
                                     Multiply, Reshape, BatchNormalization, GlobalAveragePooling1D,
                                     Lambda)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
import os
# Save df_final as a .csv file
os.chdir(r'D:\sample_dataset')
df=pd.read_csv('df_final_cleaned.csv', low_memory=False)

In [3]:
df.shape

(1500000, 22)

In [4]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif

In [5]:
from sklearn.preprocessing import LabelEncoder

# Apply Label Encoding for all object-type columns
label_encoders = {}
for column in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[column] = le.fit_transform(df[column].astype(str))
    label_encoders[column] = le

In [6]:
X = df.drop('Label', axis=1)  # Drop the Label column for features
y = df['Label'] 

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [8]:
# Encode the labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_encoded = pd.get_dummies(y_encoded).values  # One-hot encode for multiclass classification

In [9]:
# Define CNN Model
def create_cnn_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define LSTM Model
def create_lstm_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = LSTM(64, return_sequences=True)(inputs)
    x = BatchNormalization()(x)
    x = LSTM(32)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define FNN Model
def create_fnn_model(input_shape, num_classes):
    inputs = Input(shape=(input_shape,))
    x = Dense(128, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model

In [10]:
from memory_profiler import memory_usage

In [11]:
def train_ensemble():
    return ensemble_model.fit(
        [X_train_cnn, X_train_cnn, X_train], y_train,
        epochs=10,
        batch_size=128,
        validation_split=0.1,
        verbose=0,
        validation_data=([X_test_cnn, X_test_cnn, X_test], y_test)
    )

In [16]:
# Convert one-hot labels back to class labels for StratifiedKFold
y_labels = np.argmax(y_encoded, axis=1)
# --------------------------------- Part 4: 5-Fold Cross-Validation Setup ---------------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ensemble_accs, cnn_accs, lstm_accs, fnn_accs = [], [], [], []
ensemble_times, cnn_times, lstm_times, fnn_times = [], [], [], []
ensemble_memory_usages, cnn_memory_usages, lstm_memory_usages, fnn_memory_usages = [], [], [], []
all_y_test = []
all_ensemble_pred = []
all_cnn_pred=[]
all_lstm_pred=[]
all_fnn_pred=[]
all_ensemble_prob = []
all_cm_ensemble = []
all_cm_cnn = []
all_cm_lstm = []
all_cm_fnn = []
attention_weights=[]
avg_atts=[]
fold = 1
for train_idx, test_idx in kfold.split(X, y_labels):
    print(f"\n=== Fold {fold} ===")
    fold += 1

    X_train, X_test = X[train_idx], X[test_idx]

    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    # Number of classes in the dataset
    num_classes = y_train.shape[1]
    # CNN
    cnn_model = create_cnn_model(X_train_cnn.shape[1:], num_classes)
    cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    cnn_mem_usage, cnn_history = memory_usage(
    (cnn_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True )
    #cnn_history = cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    cnn_times.append(end - start)
    cnn_peak_memory = max(cnn_mem_usage)
    cnn_memory_usages.append(cnn_peak_memory)
    cnn_pred = (cnn_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    cnn_accs.append(accuracy_score(y_test, cnn_pred))
    
    # LSTM
  
    lstm_model = create_lstm_model(X_train_cnn.shape[1:], num_classes)
    lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    lstm_mem_usage, lstm_history = memory_usage(
    (lstm_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
     interval=0.1,retval=True)
    #lstm_history= lstm_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    lstm_times.append(end - start)
    lstm_peak_memory = max(lstm_mem_usage)
    lstm_memory_usages.append(lstm_peak_memory)
    lstm_pred = (lstm_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    lstm_accs.append(accuracy_score(y_test, lstm_pred))
    
     # FNN
    
    fnn_model = create_fnn_model(X_train.shape[1], num_classes)
    fnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    fnn_mem_usage, fnn_history = memory_usage(
    (fnn_model.fit, (X_train, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True)
    #fnn_history=fnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    fnn_times.append(end - start)
    fnn_peak_memory = max(fnn_mem_usage)
    fnn_memory_usages.append(fnn_peak_memory)
    fnn_pred = (fnn_model.predict(X_test,verbose=0) > 0.5).astype(int)
    fnn_accs.append(accuracy_score(y_test, fnn_pred))
    
     # Ensemble
   
    cnn_probs = cnn_model.predict(X_test_cnn,verbose=0)
    lstm_probs = lstm_model.predict(X_test_cnn,verbose=0)
    fnn_probs = fnn_model.predict(X_test,verbose=0)

    # Static average of predicted probabilities
    ensemble_prob = (cnn_probs + lstm_probs + fnn_probs) / 3.0
    ensemble_pred = (ensemble_prob > 0.5).astype(int)

    ensemble_accs.append(accuracy_score(y_test, ensemble_pred))
    ensemble_times.append(cnn_times[-1] + lstm_times[-1] + fnn_times[-1])  # Total time of all models
    ensemble_memory_usages.append(max([cnn_memory_usages[-1], lstm_memory_usages[-1], fnn_memory_usages[-1]]))  # Peak of three
    all_y_test.append(y_test)
    all_ensemble_pred.append(ensemble_pred)
    all_ensemble_prob.append(ensemble_prob)
    


=== Fold 1 ===

=== Fold 2 ===

=== Fold 3 ===

=== Fold 4 ===

=== Fold 5 ===


In [22]:

# --------------------------------- Part 5: Report Results ---------------------------------
def report_scores(name, scores, times, memories):
    print(f"{name}: Accuracy = {np.mean(scores):.4f} ± {np.std(scores):.4f}, "
          f"Time = {np.mean(times):.2f}s ± {np.std(times):.2f}s, "
          f"Memory = {np.mean(memories):.2f} MiB ± {np.std(memories):.2f} MiB")

print("\n=== 5-Fold Cross-validation Results ===")
report_scores("CNN", cnn_accs, cnn_times, cnn_memory_usages)
report_scores("LSTM", lstm_accs, lstm_times, lstm_memory_usages)
report_scores("FNN", fnn_accs, fnn_times, fnn_memory_usages)
report_scores("Ensemble", ensemble_accs, ensemble_times, ensemble_memory_usages)


=== 5-Fold Cross-validation Results ===
CNN: Accuracy = 0.8713 ± 0.0010, Time = 6144.72s ± 221.57s, Memory = 1582.64 MiB ± 67.96 MiB
LSTM: Accuracy = 0.8854 ± 0.0046, Time = 26799.18s ± 1310.11s, Memory = 1641.24 MiB ± 56.59 MiB
FNN: Accuracy = 0.8858 ± 0.0105, Time = 3840.27s ± 84.67s, Memory = 1611.84 MiB ± 68.43 MiB
Ensemble: Accuracy = 0.8797 ± 0.0035, Time = 36784.18s ± 1525.64s, Memory = 1652.21 MiB ± 47.61 MiB


In [23]:
# --------------------------------- Part 6: Statistical Significance Testing ---------------------------------
print("\n=== Paired t-tests ===")
print("Ensemble vs CNN:", ttest_rel(ensemble_accs, cnn_accs))
print("Ensemble vs LSTM:", ttest_rel(ensemble_accs, lstm_accs))
print("Ensemble vs FNN:", ttest_rel(ensemble_accs, fnn_accs))


=== Paired t-tests ===
Ensemble vs CNN: TtestResult(statistic=4.821697254614428, pvalue=0.008512503819554873, df=4)
Ensemble vs LSTM: TtestResult(statistic=-2.1794494717703428, pvalue=0.09480282578973333, df=4)
Ensemble vs FNN: TtestResult(statistic=-1.6751668609623287, pvalue=0.16921183846041585, df=4)


In [24]:
class_names =  [
    'Normal','fuzzing', 'http-flood', 'http-loris', 'http-smuggle', 'http2-concurrent','http2-pause','quic-enc','quic-flood', 
      'quic-loris'
]

In [25]:
report = classification_report(y_true_classes , y_pred_classes_ensemble, target_names=class_names)
print("Ablation-Static Ensemble without WGANGP and IMOA - Ensemble Model (5-Fold CV):H23Q-Multiclass Classification")
print(report)


Ablation-Static Ensemble without WGANGP and IMOA - Ensemble Model (5-Fold CV):H23Q-Multiclass Classification
                  precision    recall  f1-score   support

          Normal       0.99      0.89      0.94   1372539
         fuzzing       0.36      0.88      0.52      8046
      http-flood       0.91      0.92      0.92     62383
      http-loris       0.50      0.82      0.62     17502
    http-smuggle       0.03      0.98      0.05       508
http2-concurrent       0.08      0.51      0.13      8980
     http2-pause       0.04      0.20      0.06      7458
        quic-enc       0.11      0.90      0.19      1663
      quic-flood       0.65      0.88      0.75     17159
      quic-loris       0.32      0.69      0.44      3762

        accuracy                           0.88   1500000
       macro avg       0.40      0.77      0.46   1500000
    weighted avg       0.97      0.88      0.92   1500000

